In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# operation fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. The cap is inherited by the fork-based run_parallel workers,
# which share the parent frame copy-on-write, so it stays compatible with them.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 8.9G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.append(os.path.abspath("../2_Features_build")) ; import target_features
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")); from parallel_compute import *

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path
import joblib


### Evaluate the trained spike-aware ensemble model on the test set.

#### Method

Loads the trained model components, generates predictions on held-out test data, 
and reports comprehensive performance metrics including:

- MAE, RMSE, R², MBE, WMAPE across all horizons
- Spike-specific metrics (MAE for price > $150/MWh)
- Dip-specific metrics (MAE for price < $0/MWh)
- Per-horizon breakdown showing prediction accuracy vs lead time

Inputs

In [3]:
SELECTED_FEATURES_DIR = variables.CWD / "4_Features_select" / "Selected_features"
FEATURE_OBJECTIVES = ["normal", "arcsinh", "spikes", "dips"]
features_optimal_amount_by_objective = {
    objective: pd.read_parquet(SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet")
    for objective in FEATURE_OBJECTIVES
}

# The full matrix is ~10 GB and won't fit in RAM; only the selected features are
# used at predict time, so read the union of the selected columns instead.
_horizons = [f"h{i}" for i in range(1, variables.HORIZON_COUNT + 1)]
needed_features = sorted({
    feat
    for optimal in features_optimal_amount_by_objective.values()
    for h in _horizons
    for feat in optimal.loc[optimal[h] == True, "feature"].tolist()
})

features = read_parquet_float32(variables.FEATURES_DATASET_PATH, columns=needed_features)
targets  = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)


Loading..: 100%|██████████| 133/133 [00:18<00:00,  7.24batch/s]


In [4]:
import psutil

# Prediction now runs sequentially (no fork workers), so each model may use all
# cores - multi-threaded predict without the old worker x core oversubscription.
_predict_threads = max(1, psutil.cpu_count(logical=False) - 2)

trained_models = []
for path in sorted(Path(variables.TRAINED_MODELS_PATH).glob("*.joblib")):
    filename = path.stem
    parts = filename.split("_", 1)
    horizon = int(parts[0][1:])
    model_name = parts[1]
    model = joblib.load(path)
    if hasattr(model, "set_params"):
        model.set_params(n_jobs=_predict_threads, num_threads=_predict_threads)
    trained_models.append({"horizon": horizon, "model_name": model_name, "model": model})

display(trained_models[:2])


[{'horizon': 1,
  'model_name': 'full_range_regressor_clipped_MAE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                force_col_wise=True, learning_rate=0.025, max_bin=127,
                metric='mae', min_child_samples=25, n_estimators=1400, n_jobs=10,
                num_leaves=95, num_threads=10, objective='regression_l1',
                path_smooth=0.1, random_state=42, reg_alpha=0.05, reg_lambda=0.2,
                verbose=-1)},
 {'horizon': 1,
  'model_name': 'full_range_regressor_unclipped_RMSE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                force_col_wise=True, learning_rate=0.025, max_bin=127,
                metric='rmse', min_child_samples=25, n_estimators=1200, n_jobs=10,
                num_leaves=95, num_threads=10, objective='regression',
                path_smooth=0.1, random_state=43, reg_alpha=0.05, reg_lambda=0.2,
                verbose=-1)}]

In [5]:
def build_model_dict(trained_models):
    """Build model dictionary from individually saved trained models."""
    horizons = sorted(set(m["horizon"] for m in trained_models))
    mm = {(m["horizon"], m["model_name"]): m["model"] for m in trained_models}
    return {
        "full_range_regressor_clipped_MAE_loss": [mm.get((h, "full_range_regressor_clipped_MAE_loss")) for h in horizons],
        "full_range_regressor_unclipped_RMSE_loss": [mm.get((h, "full_range_regressor_unclipped_RMSE_loss")) for h in horizons],
        "positive_spike_classifier_binary_loss": [mm.get((h, "positive_spike_classifier_binary_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_mae_loss": [mm.get((h, "positive_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_quantile_loss": [mm.get((h, "positive_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "negative_spike_classifer_unclipped_binary_loss": [mm.get((h, "negative_spike_classifer_unclipped_binary_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_mae_loss": [mm.get((h, "negative_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_quantile_loss": [mm.get((h, "negative_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "horizon_list":  horizons,
    }

model_dict = build_model_dict(trained_models)

# Load optimized parameters (calibrators, blend alphas) from final_params
# This file contains ONLY optimization parameters, NOT model objects (models are loaded separately above)
final_params = joblib.load(variables.FINAL_PARAMS_PATH)
model_dict.update(final_params)

pd.DataFrame({k: v for k, v in model_dict.items() if k != "calibrators"})[:3]

,full_range_regressor_clipped_MAE_loss,full_range_regressor_unclipped_RMSE_loss,positive_spike_classifier_binary_loss,positive_spike_regressor_unclipped_mae_loss,positive_spike_regressor_unclipped_quantile_loss,negative_spike_classifer_unclipped_binary_loss,negative_spike_regressor_unclipped_mae_loss,negative_spike_regressor_unclipped_quantile_loss,horizon_list,full_range_blend_alphas,spike_adjustment_modes,spike_source_modes,spike_probability_thresholds,spike_probability_powers,spike_max_uplift_weights,dip_adjustment_modes,dip_source_modes,dip_probability_thresholds,dip_probability_powers,dip_max_reduction_weights
0,"LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMClassifier(bagging_fraction=0.85, bagging_...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.9, bagging_fraction=0.85...","LGBMClassifier(bagging_fraction=0.85, bagging_...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.1, bagging_fraction=0.85...",1,0.612245,1.0,2.0,0.005,18.0,5.0,0,1,0.005,18.0,5.0


Helpers

In [6]:
def _split(data):
    train = data[(data.index >= variables.TRAIN_START) & (data.index <= variables.VALID_START)]
    validate = data[(data.index > variables.VALID_START) & (data.index <= variables.TEST_START)]
    test  = data[data.index > variables.TEST_START]
    return train, validate, test


def convert_from_asinh(y):
    return np.sinh(y) * variables.PRICE_TRANSFORM_SCALE


def _apply_spike_policy(y_base, spike_prob, spike_reg, spike_qreg, kind, p1, p2, p3, src):
    """
    Apply spike adjustment policy.
    kind: 0 = probability-weighted blend, 1 = threshold-based uplift, 2 = gate-based
    src: 0 = use quantile, 1 = max of reg and quantile, 2 = use regressor
    p1: probability threshold
    p2: probability power
    p3: max uplift weight
    """
    # Choose spike source
    if src == 0:
        spike_source = spike_qreg
    elif src == 1:
        spike_source = np.maximum(spike_reg, spike_qreg).astype(np.float32)
    else:
        spike_source = spike_reg
    
    # Apply spike adjustment
    if kind == 0:
        # Probability-weighted blend
        prob_no_spike = 1.0 - spike_prob
        prob_weighted_spike = spike_prob * spike_source
        return (prob_no_spike * y_base + prob_weighted_spike).astype(np.float32)
    
    elif kind == 1:
        # Threshold-based uplift
        spike_prob_above_thresh = spike_prob - p1
        max_spike_prob = 1.0 - p1 + 1e-6
        normalized = np.clip(spike_prob_above_thresh / max_spike_prob, 0.0, 1.0)
        shaped_conf = np.power(normalized, p2)
        weighted_conf = (shaped_conf * p3).astype(np.float32)
        pos_spike_gap = np.maximum(spike_source - y_base, 0.0)
        uplift = pos_spike_gap * weighted_conf
        return (y_base + uplift).astype(np.float32)
    
    elif kind == 2:
        # Gate-based
        gate = spike_prob >= p1
        y_out = y_base.copy()
        shifted_base = y_base[gate] + float(p2)
        spike_gap = spike_source[gate] - y_base[gate]
        pos_gap = np.maximum(spike_gap, 0.0)
        y_out[gate] = shifted_base * pos_gap
        return y_out.astype(np.float32)
    
    return y_base


def _apply_dip_policy(y_pred, dip_prob, dip_reg, kind, p1, p2, p3):
    """
    Apply dip adjustment policy.
    kind: 0 = probability-weighted blend, 1 = threshold-based reduction
    p1: probability threshold
    p2: probability power
    p3: max reduction weight
    """
    if kind == 0:
        # Probability-weighted blend
        prob_no_dip = 1.0 - dip_prob
        prob_weighted_dip = dip_prob * dip_reg
        return (prob_no_dip * y_pred + prob_weighted_dip).astype(np.float32)
    
    elif kind == 1:
        # Threshold-based reduction
        dip_prob_above_thresh = dip_prob - p1
        max_dip_prob = 1.0 - p1 + 1e-6
        normalized = np.clip(dip_prob_above_thresh / max_dip_prob, 0.0, 1.0)
        shaped_conf = np.power(normalized, p2)
        weighted_conf = (shaped_conf * p3).astype(np.float32)
        neg_dip_gap = np.maximum(y_pred - dip_reg, 0.0)
        reduction = neg_dip_gap * weighted_conf
        return (y_pred - reduction).astype(np.float32)
    
    return y_pred

Core logic

In [7]:
from tqdm.auto import tqdm


def generate_predictions_per_horizon(task):
    """Generate predictions for a single horizon on test set."""
    (i, h, X_te, y_true, models_tuple, params_tuple) = task
    
    # Unpack model components
    (full_range_clipped, full_range_unclipped, blend_alpha,
     pos_spike_clf, pos_spike_reg_mae, pos_spike_reg_q,
     neg_spike_clf, neg_spike_reg_mae, calibrator) = models_tuple
    
    # Unpack parameters
    (spike_mode, spike_src, spike_p1, spike_p2, spike_p3,
     dip_mode, dip_p1, dip_p2, dip_p3) = params_tuple

    # X_te is a per-objective dict; each model predicts on its own feature set.
    # Base prediction: blend L1 (normal) and L2 (arcsinh) models
    y_pred_blend = convert_from_asinh(full_range_clipped.predict(X_te["normal"])).astype(np.float32)
    y_pred_blend = ((1.0 - blend_alpha) * y_pred_blend + 
                    blend_alpha * convert_from_asinh(full_range_unclipped.predict(X_te["arcsinh"]))).astype(np.float32)
    
    # Apply spike adjustment (spikes feature set)
    y_spike_prob = pos_spike_clf.predict_proba(X_te["spikes"])[:, 1].astype(np.float32)
    y_spike_mae = convert_from_asinh(pos_spike_reg_mae.predict(X_te["spikes"])).astype(np.float32)
    y_spike_q = convert_from_asinh(pos_spike_reg_q.predict(X_te["spikes"])).astype(np.float32)
    y_pred = _apply_spike_policy(y_pred_blend, y_spike_prob, y_spike_mae, y_spike_q, 
                                  spike_mode, spike_p1, spike_p2, spike_p3, spike_src)
    
    # Apply dip adjustment (dips feature set)
    y_dip_prob = neg_spike_clf.predict_proba(X_te["dips"])[:, 1].astype(np.float32)
    y_dip_mae = convert_from_asinh(neg_spike_reg_mae.predict(X_te["dips"])).astype(np.float32)
    y_pred = _apply_dip_policy(y_pred, y_dip_prob, y_dip_mae, 
                                dip_mode, dip_p1, dip_p2, dip_p3)
    
    # Apply isotonic calibration
    y_pred = calibrator.predict(y_pred.astype(np.float64)).astype(np.float32)
    
    return y_pred, y_true


def generate_predictions(model_dict, features_optimal_amount_by_objective, feature_data, target_data):
    """Generate predictions on test set using the trained model ensemble."""
    # Reuse the feature_data/target_data already loaded at notebook scope instead of
    # re-reading the ~9GB parquet files again here (was doubling parent-process memory).
    _, _, test_feat = _split(feature_data)
    _, _, test_tgt = _split(target_data)

    horizon_list = model_dict["horizon_list"]
    n_h = len(horizon_list)
    
    # Truncate test set to exclude rows without complete future data for all horizons.
    # Features are 5-min but horizons step at 30-min: horizon h needs step * h future 5-min
    # rows, so the largest horizon needs step * max(horizon_list) trailing rows dropped (NaN).
    step = variables.HORIZON_GRANULARITY_IN_MINUTES // variables.FEATURE_GRANULARITY_IN_MINUTES
    max_shift = step * max(horizon_list)
    test_tgt = test_tgt.iloc[:-max_shift]  # Remove trailing rows with incomplete future targets
    
    n_rows = len(test_tgt)

    # Extract model components
    full_range_clipped = model_dict.get("full_range_regressor_clipped_MAE_loss", [None]*n_h)
    full_range_unclipped = model_dict.get("full_range_regressor_unclipped_RMSE_loss", [None]*n_h)
    blend_alphas = model_dict.get("full_range_blend_alphas", np.zeros(n_h, dtype=np.float32))
    
    pos_spike_clf = model_dict.get("positive_spike_classifier_binary_loss", [None]*n_h)
    pos_spike_reg_mae = model_dict.get("positive_spike_regressor_unclipped_mae_loss", [None]*n_h)
    pos_spike_reg_q = model_dict.get("positive_spike_regressor_unclipped_quantile_loss", [None]*n_h)
    
    neg_spike_clf = model_dict.get("negative_spike_classifer_unclipped_binary_loss", [None]*n_h)
    neg_spike_reg_mae = model_dict.get("negative_spike_regressor_unclipped_mae_loss", [None]*n_h)
    
    spike_modes = model_dict.get("spike_adjustment_modes", np.zeros(n_h, dtype=np.int8))
    spike_srcs = model_dict.get("spike_source_modes", np.zeros(n_h, dtype=np.int8))
    spike_p1s = model_dict.get("spike_probability_thresholds", np.full(n_h, 0.20, dtype=np.float32))
    spike_p2s = model_dict.get("spike_probability_powers", np.full(n_h, 2.0, dtype=np.float32))
    spike_p3s = model_dict.get("spike_max_uplift_weights", np.full(n_h, 0.75, dtype=np.float32))
    
    dip_modes = model_dict.get("dip_adjustment_modes", np.zeros(n_h, dtype=np.int8))
    dip_p1s = model_dict.get("dip_probability_thresholds", np.full(n_h, 0.15, dtype=np.float32))
    dip_p2s = model_dict.get("dip_probability_powers", np.full(n_h, 1.0, dtype=np.float32))
    dip_p3s = model_dict.get("dip_max_reduction_weights", np.full(n_h, 0.60, dtype=np.float32))
    
    calibrators = model_dict.get("calibrators", [None]*n_h)

    # Pre-process test data for each horizon (in main process to avoid redundant loading)
    base_te = test_feat.reindex(test_tgt.index)
    TASKS = []
    for i, h in enumerate(tqdm(horizon_list, desc="Preparing test data", unit="horizon")):
        # One feature matrix per objective (selected columns + horizon-h target-time block),
        # matching the training layout so each model gets its own correctly-aligned columns.
        X_te = {}
        for objective, optimal in features_optimal_amount_by_objective.items():
            feat_cols = [c for c in optimal.loc[optimal[f"h{h}"] == True, "feature"].tolist()
                         if c in test_feat.columns]
            X_te[objective] = target_features.append_target_time_feats(base_te[feat_cols].astype(np.float32), h)
        y_true = test_tgt[f"target_h{h}"].values.astype(np.float32)
        
        models_tuple = (
            full_range_clipped[i], full_range_unclipped[i], float(blend_alphas[i]),
            pos_spike_clf[i], pos_spike_reg_mae[i], pos_spike_reg_q[i],
            neg_spike_clf[i], neg_spike_reg_mae[i], calibrators[i]
        )
        params_tuple = (
            int(spike_modes[i]), int(spike_srcs[i]), float(spike_p1s[i]), float(spike_p2s[i]), float(spike_p3s[i]),
            int(dip_modes[i]), float(dip_p1s[i]), float(dip_p2s[i]), float(dip_p3s[i])
        )
        TASKS.append((i, h, X_te, y_true, models_tuple, params_tuple))
    
    # Predict per horizon sequentially; models are multi-threaded (no fork workers).
    results = [generate_predictions_per_horizon(task)
               for task in tqdm(TASKS, desc="Predicting test horizons", unit="horizon")]
    
    # Collect results into matrices
    preds_m = np.full((n_rows, n_h), np.nan, dtype=np.float32)
    trues_m = np.full((n_rows, n_h), np.nan, dtype=np.float32)
    
    for i, (y_pred, y_true) in enumerate(results):
        preds_m[:, i] = y_pred
        trues_m[:, i] = y_true

    return preds_m, trues_m, test_tgt.index, horizon_list


/home/daniel-davaris/venv_main/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
def calculate_error_metrics(combined_df, horizon_list):
    """Calculate comprehensive evaluation metrics from predictions DataFrame."""
    
    # Helper functions for metric calculation
    def _rmse(yt, yp): 
        return float(np.sqrt(mean_squared_error(yt, yp)))
    
    def _wmape(yt, yp): 
        return float(np.sum(np.abs(yt - yp)) / (np.sum(np.abs(yt)) + 1e-8) * 100)
    
    def _mbe(yt, yp): 
        return float(np.mean(yp - yt))
    
    def _build_metrics(yt, yp):
        return {
            "mae": float(mean_absolute_error(yt, yp)), 
            "rmse": _rmse(yt, yp), 
            "r2": float(r2_score(yt, yp)), 
            "mbe": _mbe(yt, yp), 
            "wmape": _wmape(yt, yp)
        }

    # Extract actuals and predictions from combined_df
    actual_cols = [f"actual_h{h}" for h in horizon_list]
    predicted_cols = [f"predicted_h{h}" for h in horizon_list]
    
    trues_m = combined_df[actual_cols].values
    preds_m = combined_df[predicted_cols].values
    
    # Calculate overall metrics across all horizons
    yt_flat = trues_m.ravel()
    yp_flat = preds_m.ravel()
    model_metrics = _build_metrics(yt_flat, yp_flat)
    
    # Calculate spike-specific metrics (price > $150/MWh)
    spike_mask = yt_flat > variables.SPIKE_THRESHOLD
    if spike_mask.sum() > 0:
        model_metrics["spike_mae"] = float(mean_absolute_error(yt_flat[spike_mask], yp_flat[spike_mask]))
        model_metrics["nonspike_mae"] = float(mean_absolute_error(yt_flat[~spike_mask], yp_flat[~spike_mask]))
        model_metrics["spike_pct"] = float(spike_mask.mean() * 100)
    
    # Calculate dip-specific metrics (price < $0/MWh)
    dip_mask = yt_flat < variables.DIP_THRESHOLD
    if dip_mask.sum() > 0:
        model_metrics["dip_mae"] = float(mean_absolute_error(yt_flat[dip_mask], yp_flat[dip_mask]))
        model_metrics["dip_pct"] = float(dip_mask.mean() * 100)

    # Calculate per-horizon metrics
    step_records = []
    for i, h in enumerate(horizon_list):
        # Filter out NaN values for this specific horizon
        yt_h = trues_m[:, i]
        yp_h = preds_m[:, i]
        valid_h = ~np.isnan(yt_h) & ~np.isnan(yp_h)
        step_records.append({
            "step": i + 1,
            "h": h,
            "lead_h": round(h * variables.HORIZON_GRANULARITY_IN_MINUTES / 60, 1),
            "mae": round(float(mean_absolute_error(trues_m[:, i], preds_m[:, i])), 2),
            "rmse": round(_rmse(trues_m[:, i], preds_m[:, i]), 2),
            "r2": round(float(r2_score(trues_m[:, i], preds_m[:, i])), 4),
            "mbe": round(_mbe(trues_m[:, i], preds_m[:, i]), 2)
        })

    return {
        "metrics": model_metrics, 
        "results_df": combined_df, 
        "steps_df": pd.DataFrame(step_records)
    }

In [9]:
def present_pretty_error_summary(eval_output):
    """Format and display evaluation metrics in a human-readable format."""
    
    # Format overall metrics as DataFrame
    metrics = eval_output["metrics"]
    unitless_metrics = {"r2"}
    percentage_metrics = {"mape", "wmape", "spike_pct", "dip_pct"}
    
    metrics_data = []
    for metric_name, value in metrics.items():
        if metric_name in unitless_metrics:
            formatted_value = f"{value:.4f}"
            unit = ""
        elif metric_name in percentage_metrics:
            formatted_value = f"{value:.2f}"
            unit = "%"
        else:
            formatted_value = f"{value:.2f}"
            unit = "$/MWh"
        
        metrics_data.append({
            "Metric": metric_name.upper(),
            "Value": formatted_value,
            "Unit": unit
        })
    
    metrics_df = pd.DataFrame(metrics_data)
    print("\n=== Overall Metrics ===")
    print(metrics_df.to_string(index=False))
    
    # Format per-horizon breakdown
    steps_df = eval_output.get("steps_df")
    if steps_df is not None:
        display_df = steps_df.copy()
        display_df = display_df.rename(columns={
            "step": "Step",
            "lead_h": "Lead (h)",
            "mae": "MAE",
            "rmse": "RMSE",
            "r2": "R²",
            "mbe": "MBE"
        })
        display_df = display_df.drop(columns=["h"])
        
        print("\n=== Per-Horizon Performance ===")
        print(display_df.to_string(index=False))
        
        # Add summary statistics
        print(f"\n=== Summary Statistics ===")
        print(f"Best MAE:  {display_df['MAE'].min():.2f} $/MWh at step {display_df.loc[display_df['MAE'].idxmin(), 'Step']:.0f}")
        print(f"Worst MAE: {display_df['MAE'].max():.2f} $/MWh at step {display_df.loc[display_df['MAE'].idxmax(), 'Step']:.0f}")
        print(f"Best R²:   {display_df['R²'].max():.4f} at step {display_df.loc[display_df['R²'].idxmax(), 'Step']:.0f}")
        print(f"Worst R²:  {display_df['R²'].min():.4f} at step {display_df.loc[display_df['R²'].idxmin(), 'Step']:.0f}")

Execute evaluation

In [10]:
preds_m, trues_m, test_idx, horizon_list = generate_predictions(model_dict, features_optimal_amount_by_objective, features, targets)
# Export raw predictions and actuals
predictions_df = pd.DataFrame(preds_m, index=test_idx, columns=[f"predicted_h{h}" for h in horizon_list])
actuals_df = pd.DataFrame(trues_m, index=test_idx, columns=[f"actual_h{h}" for h in horizon_list])
combined_df = pd.concat([actuals_df, predictions_df], axis=1)
combined_df.insert(0, 'date', combined_df.index)
variables.ACTUAL_VS_PREDICTED_TEST_SET.parent.mkdir(parents=True, exist_ok=True)
combined_df.to_parquet(variables.ACTUAL_VS_PREDICTED_TEST_SET)
print(f"Saved evaluation predictions to {variables.ACTUAL_VS_PREDICTED_TEST_SET}")


Predicting test horizons: 100%|██████████| 1/1 [00:09<00:00,  9.67s/horizon]

Saved evaluation predictions to /home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/5_model_results/actual_vs_predicted_test_set.parquet


In [11]:
combined_df = pd.read_parquet(variables.ACTUAL_VS_PREDICTED_TEST_SET)
eval_output = calculate_error_metrics(combined_df, horizon_list)
present_pretty_error_summary(eval_output)


=== Overall Metrics ===
      Metric  Value  Unit
         MAE  35.71 $/MWh
        RMSE 283.96 $/MWh
          R2 0.4625      
         MBE  11.52 $/MWh
       WMAPE  33.11     %
   SPIKE_MAE 151.98 $/MWh
NONSPIKE_MAE  16.07 $/MWh
   SPIKE_PCT  14.45     %
     DIP_MAE  26.67 $/MWh
     DIP_PCT  13.24     %

=== Per-Horizon Performance ===
 Step  Lead (h)   MAE   RMSE     R²   MBE
    1       0.5 35.71 283.96 0.4625 11.52

=== Summary Statistics ===
Best MAE:  35.71 $/MWh at step 1
Worst MAE: 35.71 $/MWh at step 1
Best R²:   0.4625 at step 1
Worst R²:  0.4625 at step 1


In [12]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 29 variable(s); kernel rss 0.30G, 8.7G RAM free now
